In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Preset values

year = 2025

res_prop = "Residential"
res_sub = "SingleFamilyResidence"
attrs = ["ClosePrice", "LivingArea", "BedroomsTotal", "BathroomsTotalInteger", "LotSizeArea"]

bins=100

new_attrs = ["ClosePrice", "LivingArea", "Bedrooms", "Bathrooms", "LotSizeArea"]

In [ ]:
def load_data(yr=year):
  """
  Takes year of CRMLS files
  Returns merged DataFrame
  """
  main_df = pd.DataFrame()        # Defines blank DataFrame
  for i in range(1,13):           # Iterates through 12 months
      if i < 10:                    # If i is a single-digit value, add string of zero in front of the string of i
          month = "0"+str(i)
      else:                         # Else, month is defined as the string of i
          month = str(i)
      try:
          df = pd.read_csv("CRMLSSold"+str(yr)+month+".csv", low_memory=False)          # Reads csv
          main_df = pd.concat([main_df, df], ignore_index=True)                         # Appends DataFrame of csv to "main_df"
      except:
          df = pd.read_csv("CRMLSSold"+str(yr)+month+"_filled.csv", low_memory=False)
          main_df = pd.concat([main_df, df], ignore_index=True)
  return main_df

In [ ]:
def get_subset(df, prop=res_prop, sub_prop=res_sub, cols=attrs):
  """
  Takes the DataFrame, PropertyType value (str), PropertySubType value (str), and list of columns for subset DataFrame
  Returns subset of inputted DataFrame, based on inputted parameters
  """
  df = df[(df["PropertyType"] == prop)&(df["PropertySubType"] == sub_prop)]     # Takes subset of DataFrame based on property type and subtype
  sub_df = df[cols].reset_index(drop=True)                                      # Takes subset of columns for DataFrame
  return sub_df

In [ ]:
# Correlation Matrix

def make_corr(df):
  matrix = df.corr(numeric_only=True)         # Computes correlation matrix
  fig, ax = plt.subplots(figsize=(7,5))
  ax.imshow(matrix)                           # Plots visualization of correlation matrix

  vals = []
  for a in range(len(matrix)):
    for b in matrix:
      vals.append(round(float(matrix[b].iloc[a]),2))    # Appends the rounded correlation values of the matrix to list "vals"
  for c in range(len(matrix)):
    for d in range(len(matrix)):
      num = vals[d + len(matrix)*c]                     # Takes values by row and column
      if num >= .5:                                                               # If correlation is greater than or equal to 0.5
        txt = ax.text(x=(d-.28), y=(c+.05), s=num, fontsize=7.5, color="black")     # Add correlation value in black text
      else:                                                                       # Else
        txt = ax.text(x=(d-.28), y=(c+.05), s=num, fontsize=7.5, color="white")     # Add correlation value in white text
  lbls = [col for col in matrix]                                                  # Defines list of column titles as "lbls"
  ax.set_xticks(range(len(lbls)), labels=lbls, rotation=45)                       # Labels x-ticks
  ax.set_yticks(range(len(lbls)), labels=lbls)                                    # Labels y-ticks
  ax.set_title("Correlation of Key Housing Features")                             # Sets title for correlation matrix
  plt.show()

In [ ]:
# Histogram

def make_hist(df, bin_num=bins):
  fig, ax = plt.subplots(1,5,figsize=(14,2))        # Plot 1 row of 5 plots
  for col in range(df.shape[1]):                    # For each column number in the DataFrame
    ax[col].hist(df.iloc[:,col], bins=bin_num)        # For column number "col", plot the data with inputted bin number "bin_num"
  plt.show()

In [ ]:
# Box Plot

def make_boxp(df):
  for col in range(df.shape[1]):            # For each column number in the DataFrame
    df.iloc[:,col].plot.box()                 # Create a boxplot for each column "col"
    plt.show()

In [ ]:
def main():

  main = load_data()
  sub = get_subset(main)
  make_corr(sub)
  print("Some LotSizeArea values are set to 0 or null, which skews the correlation data.")
  print('''This issue will be addressed when the data is cleaned and processed.

  ''')
  make_hist(sub)
  print('''Once again, the uncleaned DataFrame seems to have led to a skew in the distribution of the data.

  ''')
  make_boxp(sub)
  print("Zero values alongside higher values of high-priced homes have led to significant presentation of outliers.")
  print('''However, outliers will likely remain after data cleaning because the box plot displays differences in higher
            versus lower-valued home, which could be further distinguished based on city, zipcode, county, etc.''')

In [ ]:
if __name__ == "__main__":
  main()